# CIFAR-10 extension v2 (stronger) — universal: Apple M1 / NVIDIA / CPU

Same controlled protocol on **CIFAR-10** (matched-noise schedule, DDPM/DDIM, seeds), now with a **standard Frechet Inception Distance** (`scipy.linalg.sqrtm`, the pytorch-fid computation) on 2048-dim Inception features. Auto-detects CUDA -> Apple **MPS** -> CPU.

## Setup (once)
Python **3.11 or 3.12** (not 3.13/3.14). Then, in Terminal:
```
pip install torch torchvision diffusers scipy numpy jupyter
```
Mac check: `python3 -c "import torch;print(torch.backends.mps.is_available())"` -> `True`.

## Two quality presets (choose in Config)
- **LIGHT** (default): 30 epochs, 2 seeds, 3 step-counts, FID from 2000 samples. Feasible on a base M1 / free T4.
- **FULL** (stronger, recommended if the GPU allows): 60 epochs, 3 seeds, 5 step-counts, FID from 5000 samples. Better absolute FID, tighter error bars — more convincing at review, but several hours.

## Run order (cheapest first)
1. Anchor model (T = 1000). 2. **Experiment B** (DDPM vs DDIM across steps) — the headline. 3. **Experiment A** (FID vs T with seeds).

FID from a few thousand samples is still noisy on CIFAR (full FID uses tens of thousands); that is why we report mean +/- std. Keep the machine awake.


In [1]:
import pathlib, shutil, site, subprocess, sys

if sys.platform.startswith('linux') and shutil.which('nvidia-smi'):
    print('Installing CUDA 12.6 PyTorch wheels so Pascal GPUs (e.g. P100, sm_60) work on Kaggle.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir',
        'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])

for pkg in ('pillow', 'Pillow-SIMD'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], check=False)

for root in site.getsitepackages():
    root_path = pathlib.Path(root)
    for pattern in ('PIL', 'pillow*.dist-info', 'Pillow*.dist-info', 'Pillow_SIMD*.dist-info'):
        for path in root_path.glob(pattern):
            if path.is_dir():
                shutil.rmtree(path, ignore_errors=True)
            elif path.exists():
                path.unlink()

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-cache-dir', 'pillow==11.3.0'
])

print('Keeping Kaggle\'s bundled numpy stack and avoiding scipy/diffusers imports in the notebook body.')


Installing CUDA 12.6 PyTorch wheels so Pascal GPUs (e.g. P100, sm_60) work on Kaggle.
Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
Keeping Kaggle's bundled numpy stack and avoiding scipy/diffusers imports in the notebook body.
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


### Embedded core (auto-device, CIFAR U-Net, matched-noise schedule, samplers, standard Inception-FID)


In [2]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import math, random, json, time
import numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.models as tv_models

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)

def to_pm1(x): return x * 2.0 - 1.0

def cifar_loader(batch_size=64, train=True):
    tfm = transforms.Compose([transforms.ToTensor(), transforms.Lambda(to_pm1)])
    ds = datasets.CIFAR10(root='./data', train=train, download=True, transform=tfm)
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=train, num_workers=0, drop_last=train)

def sinusoidal_embedding(timesteps, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=timesteps.device) / half)
    args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb

class CIFARUNet(nn.Module):
    """Small pure-PyTorch U-Net for 32x32 RGB diffusion.
    This avoids the brittle diffusers import path on Kaggle while keeping
    the same unconditional DDPM training interface."""
    def __init__(self, time_dim=128):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, 256)
        )
        self.conv0 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv1 = nn.Conv2d(64, 128, 4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(128, 256, 4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(256, 256, 3, padding=1)
        self.deconv1 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.out_conv = nn.Conv2d(64, 3, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t):
        t = self.time_mlp(sinusoidal_embedding(t, self.time_dim))[:, :, None, None]
        h0 = self.act(self.conv0(x))
        h1 = self.act(self.conv1(h0))
        h2 = self.act(self.conv2(h1))
        h3 = self.act(self.conv3(h2 + t))
        u1 = self.act(self.deconv1(h3)) + h1
        u2 = self.act(self.deconv2(u1)) + h0
        return self.out_conv(u2)

def make_schedule(T, beta_1=1e-4, target_alpha_bar_T=1e-3):
    low, high = beta_1, 0.999
    for _ in range(100):
        mid = (low + high) / 2
        ab = torch.cumprod(1. - torch.linspace(beta_1, mid, T), dim=0)
        low, high = (mid, high) if ab[-1] > target_alpha_bar_T else (low, mid)
    betas = torch.linspace(beta_1, high, T, device=device)
    alphas = 1. - betas; alpha_bar = torch.cumprod(alphas, dim=0)
    return {'T': T, 'betas': betas, 'alphas': alphas, 'alpha_bar': alpha_bar,
            'sqrt_alpha_bar': torch.sqrt(alpha_bar), 'sqrt_one_minus_alpha_bar': torch.sqrt(1. - alpha_bar)}

def train_ddpm(T, epochs, loader, lr=2e-4, log=print):
    schedule = make_schedule(T); model = CIFARUNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    use_amp = (device.type == 'cuda'); scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    for ep in range(1, epochs + 1):
        model.train(); last = 0.0
        for x, _ in loader:
            x = x.to(device); t = torch.randint(0, T, (x.size(0),), device=device); noise = torch.randn_like(x)
            x_t = (schedule['sqrt_alpha_bar'][t].view(-1,1,1,1) * x +
                   schedule['sqrt_one_minus_alpha_bar'][t].view(-1,1,1,1) * noise)
            opt.zero_grad()
            if use_amp:
                with torch.autocast(device_type='cuda'):
                    loss = F.mse_loss(model(x_t, t), noise)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = F.mse_loss(model(x_t, t), noise); loss.backward(); opt.step()
            last = float(loss.item())
        if ep % 5 == 0 or ep == epochs: log(f'      epoch {ep}/{epochs} loss={last:.4f}')
    return model, schedule

def make_steps(T_train, T_infer):
    idxs = torch.linspace(0, T_train - 1, T_infer, dtype=torch.long)
    return list(reversed(idxs.tolist()))

@torch.no_grad()
def sample_ddpm(model, schedule, n, steps=None):
    model.eval(); T = schedule['T']
    if steps is None: steps = list(range(T - 1, -1, -1))
    x = torch.randn(n, 3, 32, 32, device=device)
    for i in steps:
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t); a, ab, b = schedule['alphas'][i], schedule['alpha_bar'][i], schedule['betas'][i]
        x = (1. / torch.sqrt(a)) * (x - (b / torch.sqrt(1. - ab)) * eps)
        if i > 0: x = x + torch.sqrt(b) * torch.randn_like(x)
    return x.clamp(-1., 1.)

@torch.no_grad()
def sample_ddim(model, schedule, n, steps):
    model.eval(); ab = schedule['alpha_bar']
    x = torch.randn(n, 3, 32, 32, device=device)
    for i, j in zip(steps[:-1], steps[1:]):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        x0 = (x - torch.sqrt(1. - ab[i]) * eps) / torch.sqrt(ab[i])
        x = torch.sqrt(ab[j]) * x0 + torch.sqrt(1. - ab[j]) * eps
    return x.clamp(-1., 1.)

def get_inception():
    try:
        net = tv_models.inception_v3(weights=tv_models.Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
    except Exception:
        net = tv_models.inception_v3(pretrained=True, transform_input=False)
    net.fc = nn.Identity(); net.eval(); return net.to(device)

def _prep(imgs):
    imgs = (imgs + 1.) / 2.
    imgs = F.interpolate(imgs, size=(299, 299), mode='bilinear', align_corners=False)
    mean = torch.tensor([0.485,0.456,0.406], device=imgs.device).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=imgs.device).view(1,3,1,1)
    return (imgs - mean) / std

@torch.no_grad()
def _acts(inception, source, n_max):
    feats, seen = [], 0
    if isinstance(source, DataLoader):
        for x, _ in source:
            f = inception(_prep(x.to(device))); feats.append(f.float().cpu().numpy()); seen += f.size(0)
            if seen >= n_max: break
    else:
        while seen < n_max:
            cur = min(128, n_max - seen)
            f = inception(_prep(source(cur).to(device))); feats.append(f.float().cpu().numpy()); seen += cur
    return np.concatenate(feats, 0)[:n_max]

def _sqrtm_trace_product(sigma1, sigma2, eps=1e-6):
    """Return trace(sqrt(sqrt(s1) @ s2 @ sqrt(s1))) for PSD covariance matrices.
    This is algebraically equivalent to the scipy sqrtm term in FID but uses
    only numpy.linalg, which is more robust on Kaggle's prebuilt runtime."""
    sigma1 = ((sigma1 + sigma1.T) * 0.5) + np.eye(sigma1.shape[0]) * eps
    sigma2 = ((sigma2 + sigma2.T) * 0.5) + np.eye(sigma2.shape[0]) * eps
    w1, v1 = np.linalg.eigh(sigma1)
    w1 = np.clip(w1, a_min=0.0, a_max=None)
    sqrt_sigma1 = (v1 * np.sqrt(w1)) @ v1.T
    middle = sqrt_sigma1 @ sigma2 @ sqrt_sigma1
    middle = (middle + middle.T) * 0.5
    wm = np.linalg.eigvalsh(middle)
    wm = np.clip(wm, a_min=0.0, a_max=None)
    return float(np.sqrt(wm).sum())

def _fid(mu1, sigma1, mu2, sigma2):
    """Standard Frechet Inception Distance without scipy.sqrtm dependency."""
    diff = mu1 - mu2
    tr_covmean = _sqrtm_trace_product(sigma1, sigma2)
    return float(diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2.0 * tr_covmean)

def fid_for_generator(gen_fn, real_ds, n_real, n_gen, inception):
    rl = DataLoader(real_ds, batch_size=128, shuffle=True, num_workers=0)
    rf = _acts(inception, rl, n_real); gf = _acts(inception, gen_fn, n_gen)
    return _fid(rf.mean(0), np.cov(rf, rowvar=False), gf.mean(0), np.cov(gf, rowvar=False))



### Config — pick a preset


In [3]:
PRESET = 'LIGHT'          # 'LIGHT' (fast) or 'FULL' (stronger)
if PRESET == 'FULL':
    EPOCHS = 60; SEEDS = [0, 1, 2]; STEP_COUNTS = [100, 200, 400, 700, 1000]; N_REAL = N_GEN = 5000; BATCH = 128
else:
    EPOCHS = 30; SEEDS = [0, 1];    STEP_COUNTS = [200, 500, 1000];           N_REAL = N_GEN = 2000; BATCH = 64
SAMPLER_STEPS   = [1000, 500, 200, 100, 50]
RUN_EXPERIMENT_A = True   # False = run only the fast sampler comparison (Experiment B)
# If you hit out-of-memory on a Mac, lower BATCH (e.g. 32).
inception = get_inception()
print('preset:', PRESET, '| device:', device)


preset: LIGHT | device: cuda


### Anchor model: train one CIFAR model at T = 1000 (used by Experiment B)


In [4]:
set_seed(SEEDS[0])
train_ds, train_loader = cifar_loader(batch_size=BATCH, train=True)
t0 = time.time()
anchor_model, anchor_sched = train_ddpm(1000, EPOCHS, train_loader)
print(f'anchor T=1000 trained in {(time.time()-t0)/60:.1f} min')


      epoch 5/30 loss=0.0740
      epoch 10/30 loss=0.0375
      epoch 15/30 loss=0.0498
      epoch 20/30 loss=0.0430
      epoch 25/30 loss=0.0547
      epoch 30/30 loss=0.0728
anchor T=1000 trained in 12.3 min


## Experiment B - DDPM vs DDIM FID across restoration steps (CIFAR-10)


In [5]:
table2 = {'DDPM': {}, 'DDIM': {}}
for s in SAMPLER_STEPS:
    steps = make_steps(1000, s)
    fd = fid_for_generator(lambda bs: sample_ddpm(anchor_model, anchor_sched, bs, steps=steps), train_ds, N_REAL, N_GEN, inception)
    fi = fid_for_generator(lambda bs: sample_ddim(anchor_model, anchor_sched, bs, steps),        train_ds, N_REAL, N_GEN, inception)
    table2['DDPM'][s]=fd; table2['DDIM'][s]=fi
    print(f'steps={s:>4}:  DDPM FID={fd:6.1f}   DDIM FID={fi:6.1f}')
    json.dump(table2, open('cifar_table2_sampler_fid.json','w'), indent=2)
print('\n=== CIFAR-10: sampler x steps (FID) ===')
print('steps   ' + '  '.join(f'{s:>6}' for s in SAMPLER_STEPS))
for name in ('DDPM','DDIM'):
    print(f'{name}   ' + '  '.join(f'{table2[name][s]:6.1f}' for s in SAMPLER_STEPS))


steps=1000:  DDPM FID= 111.6   DDIM FID= 116.3
steps= 500:  DDPM FID= 339.2   DDIM FID= 113.7
steps= 200:  DDPM FID= 404.7   DDIM FID= 109.3
steps= 100:  DDPM FID= 457.2   DDIM FID= 105.8
steps=  50:  DDPM FID= 494.8   DDIM FID=  99.1

=== CIFAR-10: sampler x steps (FID) ===
steps     1000     500     200     100      50
DDPM    111.6   339.2   404.7   457.2   494.8
DDIM    116.3   113.7   109.3   105.8    99.1


## Experiment A - FID vs T with uncertainty on CIFAR-10 (optional, heavier)


In [6]:
if RUN_EXPERIMENT_A:
    fid_runs = {T: [] for T in STEP_COUNTS}
    trained = {(1000, SEEDS[0]): (anchor_model, anchor_sched)}
    for seed in SEEDS:
        set_seed(seed); _, loader = cifar_loader(batch_size=BATCH, train=True)
        for T in STEP_COUNTS:
            if (T, seed) in trained:
                model, sched = trained[(T, seed)]
            else:
                set_seed(seed); print(f'training T={T} seed={seed} ...'); model, sched = train_ddpm(T, EPOCHS, loader)
            fid = fid_for_generator(lambda bs: sample_ddpm(model, sched, bs), train_ds, N_REAL, N_GEN, inception)
            fid_runs[T].append(fid); print(f'seed={seed} T={T}: FID={fid:.1f}')
    summary = {}
    print('\n=== CIFAR-10 Table 1: FID mean +/- std over seeds ===')
    for T in STEP_COUNTS:
        a = np.array(fid_runs[T]); m, sd = float(a.mean()), float(a.std(ddof=1) if len(a)>1 else 0.0)
        summary[str(T)] = (m, sd); print(f'T={T:>4}: {m:6.1f} +/- {sd:4.1f}   runs={np.round(a,1).tolist()}')
        json.dump(summary, open('cifar_table1_fid_variance.json','w'), indent=2)


training T=200 seed=0 ...
      epoch 5/30 loss=0.0493
      epoch 10/30 loss=0.0426
      epoch 15/30 loss=0.0434
      epoch 20/30 loss=0.0268
      epoch 25/30 loss=0.0494
      epoch 30/30 loss=0.0464
seed=0 T=200: FID=128.0
training T=500 seed=0 ...
      epoch 5/30 loss=0.0693
      epoch 10/30 loss=0.0470
      epoch 15/30 loss=0.0632
      epoch 20/30 loss=0.0496
      epoch 25/30 loss=0.0613
      epoch 30/30 loss=0.0666
seed=0 T=500: FID=111.8
seed=0 T=1000: FID=113.7
training T=200 seed=1 ...
      epoch 5/30 loss=0.0711
      epoch 10/30 loss=0.0432
      epoch 15/30 loss=0.0492
      epoch 20/30 loss=0.0575
      epoch 25/30 loss=0.0507
      epoch 30/30 loss=0.0419
seed=1 T=200: FID=135.0
training T=500 seed=1 ...
      epoch 5/30 loss=0.0704
      epoch 10/30 loss=0.0496
      epoch 15/30 loss=0.0713
      epoch 20/30 loss=0.0611
      epoch 25/30 loss=0.0458
      epoch 30/30 loss=0.0545
seed=1 T=500: FID=115.4
training T=1000 seed=1 ...
      epoch 5/30 loss=0.0540
   

## Send back
Send me `cifar_table2_sampler_fid.json` and (if run) `cifar_table1_fid_variance.json`, or paste the printed tables. I add a CIFAR-10 section to the manuscript and update the response letter around the actual numbers - no fabrication.

State the preset used (LIGHT/FULL), the seeds, and that FID uses the standard scipy sqrtm computation on 2048-dim Inception features - reviewers care about that detail.
